In [3]:
import pandas as pd
import joblib

seg_df = pd.read_csv("../data/processed/churn_risk_segmented.csv")

model = joblib.load("../models/best_model.pkl")

seg_df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,churn_probability,risk_segment
0,0,65,94.55,6078.75,True,True,True,True,False,True,...,False,False,False,True,False,True,False,False,0.034962,Low
1,0,26,35.75,1022.50,True,False,False,False,True,False,...,False,False,False,False,False,False,True,False,0.161273,Low
2,0,68,90.20,6297.65,False,True,False,True,False,True,...,False,False,False,True,False,True,False,False,0.038419,Low
3,0,3,84.30,235.05,True,False,False,True,False,False,...,False,True,False,False,False,False,True,False,0.567004,Medium
4,0,49,40.65,2070.75,False,True,False,False,True,False,...,False,False,False,False,False,False,False,False,0.151070,Low


In [4]:
seg_df = pd.read_csv("../data/processed/churn_risk_segmented.csv")


model = joblib.load("../models/best_model.pkl")

seg_df.head()


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,churn_probability,risk_segment
0,0,65,94.55,6078.75,True,True,True,True,False,True,...,False,False,False,True,False,True,False,False,0.034962,Low
1,0,26,35.75,1022.50,True,False,False,False,True,False,...,False,False,False,False,False,False,True,False,0.161273,Low
2,0,68,90.20,6297.65,False,True,False,True,False,True,...,False,False,False,True,False,True,False,False,0.038419,Low
3,0,3,84.30,235.05,True,False,False,True,False,False,...,False,True,False,False,False,False,True,False,0.567004,Medium
4,0,49,40.65,2070.75,False,True,False,False,True,False,...,False,False,False,False,False,False,False,False,0.151070,Low


In [7]:
feature_cols = seg_df.drop(
    ["churn_probability", "risk_segment"],
    axis=1
).columns


In [8]:
sim_contract = seg_df.copy()

if "Contract_Month-to-month" in sim_contract.columns:
    sim_contract["Contract_Month-to-month"] = 0
    sim_contract["Contract_One year"] = 1

sim_contract["new_prob"] = model.predict_proba(
    sim_contract[feature_cols]
)[:, 1]

sim_contract.to_csv(
    "../data/processed/scenario_contract.csv",
    index=False
)

print("Contract strategy saved")


Contract strategy saved


In [9]:
sim_discount = seg_df.copy()

if "MonthlyCharges" in sim_discount.columns:
    sim_discount["MonthlyCharges"] *= 0.9

sim_discount["new_prob"] = model.predict_proba(
    sim_discount[feature_cols]
)[:, 1]

sim_discount.to_csv(
    "../data/processed/scenario_discount.csv",
    index=False
)

print("Discount strategy saved")


Discount strategy saved


In [10]:
sim_support = seg_df.copy()

if "TechSupport_No" in sim_support.columns:
    sim_support["TechSupport_No"] = 0
    sim_support["TechSupport_Yes"] = 1

sim_support["new_prob"] = model.predict_proba(
    sim_support[feature_cols]
)[:, 1]

sim_support.to_csv(
    "../data/processed/scenario_support.csv",
    index=False
)

print("Tech support strategy saved")


Tech support strategy saved


In [11]:
results = pd.DataFrame({
    "Strategy": ["Original", "Contract Upgrade", "Discount", "Tech Support"],
    "Avg_Churn_Probability": [
        seg_df["churn_probability"].mean(),
        sim_contract["new_prob"].mean(),
        sim_discount["new_prob"].mean(),
        sim_support["new_prob"].mean()
    ]
})

results


,Strategy,Avg_Churn_Probability
0,Original,0.265831
1,Contract Upgrade,0.265831
2,Discount,0.255408
3,Tech Support,0.265831


In [12]:
results.to_csv(
    "../data/processed/strategy_comparison.csv",
    index=False
)

print("Strategy comparison saved")


Strategy comparison saved


## Strategy Simulation

This notebook simulates different customer retention strategies
(contract upgrade, discounts, and tech support) using the trained
churn prediction model. The goal is to evaluate which strategy
most effectively reduces predicted churn probability.
